# Eksperimen RQ1 NER (tahap 2) — Colab

RQ1: apakah **normalisasi** dan **augmentasi** teks informal memperbaiki NER pada teks informal?
Dua layer ([0012](https://github.com/danenftyessir/indonesian-hoax-checker/blob/main/docs/decisions/0012-layer-entitas-dan-claim-terpisah.md)): model **CLAIM** (F1 informal baru 0,10) dan model **entitas**.

| Varian | Perlakuan |
|---|---|
| `base` | tanpa perlakuan (pembanding) |
| `norm` | normalisasi teks informal di semua split dan saat prediksi |
| `aug` | +1 salinan informal dari data latih |
| `aug3` | +3 salinan informal (hanya CLAIM) |

Varian dan seed dibaca dari `configs/experiments/rq1_ner.yaml` (7 varian × 3 seed). Dinilai pada seed test (formal), 30 kalimat informal tulisan tangan, dan seed test versi medsos sintetis.

Cara pakai: runtime GPU, kode sudah di-push, jalankan semua sel. Model tahap 1 (NERGrit) dipulihkan dari Drive.

In [ ]:
# Pengaturan
REPO_URL = "https://github.com/danenftyessir/indonesian-hoax-checker.git"
BRANCH = "feat/ner"

# Folder Drive tim: tubes_kelompok9 > hasil_training
DRIVE_FOLDER_ID = "1Eq9C3SIW42xYpRn5UVAwoiLrRkOY52Pn"

# Model tahap 1 (NERGrit) dipulihkan dari run ini; varian RQ1 dilatih dari model itu
RESTORE_FROM_DRIVE = True
RESTORE_RUN = "ner_20261111-1312"

# Kosongkan untuk semua seed di rq1_ner.yaml; mis. "42" untuk uji cepat
SEEDS_OVERRIDE = ""

## 1. Cek GPU dan login Google (untuk upload hasil)

In [ ]:
!nvidia-smi

from google.colab import auth
auth.authenticate_user()

## 2. Ambil kode dan install dependensi

Kalau repo private, simpan token GitHub di Colab Secrets dengan nama `GITHUB_TOKEN`.

In [ ]:
import os

# Pakai token kalau repo private
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    url = REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
%cd /content/repo
!git pull -q
!git log --oneline -1
!pip install -q -e ".[training]"

## 3. Siapkan data dan pulihkan model tahap 1

`ner.prepare` menyiapkan NERGrit, `annotation split` membuat file seed (termasuk seed test versi medsos sintetis), lalu `ner_results.zip` baseline dipulihkan dari Drive.

In [ ]:
!python -m indonesian_hoax_checker.ner.prepare
!python -m indonesian_hoax_checker.ner.annotation split

In [ ]:
import io
import zipfile

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload


def find_results_zip(drive, parent_id, run_name=""):
    # Subfolder ner_* terbaru dulu (rq1_ner_* tidak ikut)
    query = f"'{parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    folders = drive.files().list(q=query, orderBy="createdTime desc", fields="files(id, name)",
                                 supportsAllDrives=True, includeItemsFromAllDrives=True).execute()["files"]
    for folder in folders:
        if not folder["name"].startswith("ner_") or (run_name and folder["name"] != run_name):
            continue
        query = f"'{folder['id']}' in parents and name = 'ner_results.zip' and trashed = false"
        files = drive.files().list(q=query, fields="files(id)", supportsAllDrives=True,
                                   includeItemsFromAllDrives=True).execute()["files"]
        if files:
            return folder["name"], files[0]["id"]
    return None, None


if RESTORE_FROM_DRIVE:
    drive = build("drive", "v3")
    run_name, file_id = find_results_zip(drive, DRIVE_FOLDER_ID, RESTORE_RUN)
    if file_id is None:
        print("Tidak ada ner_results.zip di Drive; tahap 1 akan dilatih dari awal.")
    else:
        # Unduh ke file, bukan memori: zip berisi checkpoint IndoBERT (~1,5 GB)
        with io.FileIO("/content/restore.zip", "wb") as f:
            downloader = MediaIoBaseDownload(f, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
            done = False
            while not done:
                _, done = downloader.next_chunk()
        zipfile.ZipFile("/content/restore.zip").extractall(".")
        print(f"Dipulihkan dari {run_name}")
        !ls artifacts/checkpoints/ner outputs/metrics

## 4. Jalankan semua varian

Varian yang ringkasannya sudah ada dilewati, jadi sel ini aman dijalankan ulang bila sesi terputus. Checkpoint varian dihapus setelah dinilai supaya disk tidak penuh; yang disimpan hanya metrik dan prediksi.

In [ ]:
import yaml

plan = yaml.safe_load(open("configs/experiments/rq1_ner.yaml"))
seeds = SEEDS_OVERRIDE or " ".join(str(s) for s in plan["seeds"])
names = {}
for config in plan["variants"]:
    name = yaml.safe_load(open(config))["name"]
    names[config] = name
    if os.path.exists(f"outputs/metrics/{name}_summary.json"):
        print("lewati", name)
        continue
    print("==", name, "seed:", seeds)
    !python -m indonesian_hoax_checker.ner.train --config {config} --seeds {seeds} 2>&1 | grep -E "best epoch|Error|error"
    !rm -rf artifacts/checkpoints/ner/{name}_seed*

## 5. Ringkasan RQ1 (F1, rata-rata ± simpangan baku)

`F1` = seed test formal. Kolom informal: `seed_informal_test` (30 kalimat tulisan tangan) dan `seed_sim_informal_test` (versi medsos sintetis).

In [ ]:
import json
import pandas as pd

rows = []
for config, name in names.items():
    path = f"outputs/metrics/{name}_summary.json"
    if not os.path.exists(path):
        continue
    s = json.load(open(path))
    row = {"varian": name, "F1 formal": f"{s['test_f1']['mean']:.3f} ± {s['test_f1']['std']:.3f}"}
    for key, v in s.get("extra_f1", {}).items():
        row[key] = f"{v['mean']:.3f} ± {v['std']:.3f}"
    rows.append(row)
table = pd.DataFrame(rows)
table.to_csv("/content/rq1_summary.csv", index=False)
table

## 6. Upload hasil ke Drive tim

Isi `hasil_training/rq1_ner_<waktu>/`: ringkasan tiap varian (JSON), `rq1_summary.csv`, dan `rq1_ner_results.zip` (metrik dan prediksi). Checkpoint tidak ikut supaya ukurannya kecil.

In [ ]:
from datetime import datetime

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

drive = build("drive", "v3")


def make_folder(name, parent_id):
    body = {"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]}
    return drive.files().create(body=body, fields="id", supportsAllDrives=True).execute()["id"]


def upload(path, parent_id):
    media = MediaFileUpload(path, resumable=True)
    body = {"name": os.path.basename(path), "parents": [parent_id]}
    drive.files().create(body=body, media_body=media, fields="id", supportsAllDrives=True).execute()
    print("uploaded", path)


run_folder = make_folder(f"rq1_ner_{datetime.now():%Y%m%d-%H%M}", DRIVE_FOLDER_ID)

!rm -f /content/rq1_ner_results.zip
!zip -q -r /content/rq1_ner_results.zip outputs/metrics outputs/predictions
for name in names.values():
    path = f"outputs/metrics/{name}_summary.json"
    if os.path.exists(path):
        upload(path, run_folder)
for path in ["/content/rq1_summary.csv", "/content/rq1_ner_results.zip"]:
    try:
        upload(path, run_folder)
    except Exception as err:
        print(f"GAGAL upload {path}: {err}\nKuota Drive penuh? Kosongkan Drive, lalu jalankan ulang sel ini (file masih ada di /content).")
print(f"https://drive.google.com/drive/folders/{run_folder}")